# TripAdvisor Sentiment Analysis — Step 2: Data Cleaning & Preprocessing

Welcome to the second stage of your research project! In this notebook, we will clean and preprocess the raw text data collected in Step 1.

### Learning Objectives:
1. Explain why text data cleaning (preprocessing) is critical in text mining.
2. Remove symbols, punctuation, HTML tags, numbers, emojis, and stopwords.
3. Perform **Tokenization** to break sentences into words.
4. Save clean datasets for sentiment scoring.

---

## 1. The Importance of Data Cleaning

Raw e-WOM reviews are noisy. They contain emojis, punctuation like `!!!`, extra spaces, and common functional words like *"the"*, *"is"*, and *"at"* (stopwords) that do not carry sentimental meaning. 

If we don't clean this noise, the computer will struggle to identify true sentiment patterns.

**Example:**
* *Original sentence:* `"The hotel was sooo gooood!!! 😍"`
* *After cleaning:* `"hotel good"`

Let's load the required packages (`tidyverse` and `tidytext`) and the custom R helper scripts (`helpers.R`) which contain our specific data-cleaning algorithms.

In [ ]:
# =====================================================================
# TripAdvisor Sentiment Analysis - Step 2: Data Cleaning & Preprocessing
# =====================================================================
# Welcome! If you have never programmed before, do not worry.
# We will explain exactly what the computer is doing at every single step.
#
# PURPOSE OF THIS SCRIPT:
# Text written by humans is very "messy". We use emojis, punctuation like "!!!",
# and slang like "u" instead of "you". Before a computer can understand emotions, 
# we have to scrub all this "noise" away. This is called Text Pre-Processing.

## 2. Load Raw Reviews Data

We read the raw TripAdvisor data saved from Step 1. We use `read_csv2()` to guarantee perfect column alignment for semicolon-delimited files, which is the standard format for systems configured in European and Indonesian regions.

In [ ]:
# =====================================================================
# STEP 1: Load Required Packages (Adding Tools)
# =====================================================================
# 'tidyverse' helps us filter and change table data easily.
library(tidyverse)
# 'tidytext' is a specialized toolset used purely for analyzing text and words.
library(tidytext)

# We wrote a file called 'helpers.R' that contains our secret cleaning recipes.
# 'source()' tells the computer to load our custom recipes into memory.
source("scripts/helpers.R")

cat("Helper functions and libraries loaded!\n")

## 3. Apply Text Cleaning Pipeline

We apply our custom text cleaning functions to the review text column:
1. `clean_text()`: removes HTML, numbers, extra spaces, punctuations, and lowercase the text.
2. `normalize_slang()`: converts common slang (e.g. `sooo good`, `u`) to standard dictionary words.

We then print a comparison to see how dramatically the text has been simplified.

In [ ]:
# =====================================================================
# STEP 2: Open the Raw Data
# =====================================================================
# 'read_csv2' reads the Excel file we created in Step 1 and puts it in a 
# box named 'raw_data' so we can look at it.
raw_data <- read_csv2("data/raw/bvlgari_raw_reviews.csv")

cat("Successfully loaded", nrow(raw_data), "raw reviews!\n")

## 4. Tokenization & Stopwords Removal

**Tokenization** breaks entire review sentences into individual units (usually single words, called **unigrams**). We use the `tidytext` package function `unnest_tokens()` to do this. The resulting dataframe will have 1 row per word, increasing the row count heavily.

Then, we remove standard English **stopwords** (e.g., *"the"*, *"and"*, *"is"*, *"at"*) using our helper function `remove_stopwords()`. These words occur very frequently but carry absolutely zero sentimental meaning. Removing them optimizes the dataset for the sentiment algorithms.

In [ ]:
# =====================================================================
# STEP 3: The Cleaning Pipeline
# =====================================================================
# The %>% symbol is a "pipe". It means "AND THEN".
# We are taking the raw data, AND THEN changing (mutating) the text column.
# 
# Our custom 'clean_text' recipe removes numbers, exclamation marks, and emojis.
# Our custom 'normalize_slang' recipe fixes internet slang (e.g., sooo -> so).

cleaned_reviews_df <- raw_data %>%
  mutate(
    # Create a new column called 'cleaned_text' that contains the scrubbed reviews.
    cleaned_text = clean_text(review_text),
    cleaned_text = normalize_slang(cleaned_text)
  )

# Let's print out the 2nd row to see what the computer did!
cat("--- THIS WAS THE MESSY ORIGINAL TEXT ---\n", cleaned_reviews_df$review_text[2], "\n\n")
cat("--- THIS IS THE PERFECTLY CLEANED TEXT ---\n", cleaned_reviews_df$cleaned_text[2], "\n")

## 5. Export Cleaned Datasets

We will save two datasets in `data/cleaned/`:
1. `bvlgari_cleaned_reviews.csv` — Full reviews with a column for cleaned review sentences (used for sentence scoring).
2. `bvlgari_cleaned_tokens.csv` — Tokenized reviews (one row per word) for word-frequency analysis and wordclouds.

We use `write_excel_csv2` to output semicolon-delimited CSVs for perfect formatting compatibility.

In [ ]:
# =====================================================================
# STEP 4: Tokenization (Chopping Sentences into Words)
# =====================================================================
# Computers don't read paragraphs; they read individual words.
# 'unnest_tokens' is a tool that takes a full sentence and chops it up.
# If a review is 10 words long, this tool creates 10 separate rows in our table!
# This process is formally called "Tokenization".

tokenized_df <- cleaned_reviews_df %>%
  unnest_tokens(output = word, input = cleaned_text)

cat("After chopping the sentences, we have", nrow(tokenized_df), "individual words!\n")

## 🎓 Student Exercise / Assignment Connection

For your project assignment, make sure that:
1. You understand what a stopword is and why it should be deleted. 
2. If you are scraping an Indonesian hotel/destination, you must replace the English stopword lexicons with an Indonesian stopword list (using the `stopwords` package in R with source = `"stopwords-iso"` and language = `"id"`).

**Great job! Open `03_sentiment_analysis.ipynb` to calculate sentiment scores.**